<a href="https://colab.research.google.com/github/sahilaf/Few-Shot-Cross-Device-Transfer-for-Quantum-Noise-Modeling-on-Real-Hardware/blob/main/Reproduce_quantum_noise_modeling_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ============================================================
# CELL 1 — Install Dependencies
# ============================================================
# WHEN: Run this FIRST, only once per Colab session.
# WHY:  Installs everything the notebook needs.
# ============================================================

In [ ]:
!pip install qiskit qiskit-ibm-runtime qiskit-aer numpy pandas \
             pyarrow matplotlib torch scikit-learn -q

print("✅ All dependencies installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 67.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 67.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 89.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.8/386.8 kB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.9/101.9 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 85.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.8/212.8 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.8/75.8 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 15.1 MB/s eta 0:00:00
✅ All dependencies installed


# ============================================================
# END-TO-END FROZEN Reproduce PIPELINE
# ============================================================
# USES ONLY SAVED FILES:
   - noisy_A.json
   - noisy_B.json
   - calibration_A.json
   - calibration_B.json
   - ideal.json
   - circuit_meta.json
# OUTPUT FILES:
   - dataset.json
   - dataset_standardized.json
   - model_backend_a_improved.pt
   - training_history.json
   - example_prediction.json
   - experiment_results_fixed.json
   - ablation_results_fixed.json
# ============================================================

# ============================================================
# Imports
# ============================================================

In [ ]:
import os
import json
import copy
import random
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, random_split

# ============================================================
# CONFIG
# ============================================================

In [ ]:

SEED = 42
MAX_STATES = 32        # max 2^5
SCALAR_DIM = 9
INPUT_DIM = 41         # 9 scalars + 32 noisy distribution
OUTPUT_DIM = 32

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Stronger determinism than before
try:
    torch.use_deterministic_algorithms(True)
except Exception:
    pass

if torch.cuda.is_available():
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)


Using device: cuda


# ============================================================
# REQUIRED FILES
# ============================================================

In [ ]:
required_files = [
    "noisy_A.json",
    "noisy_B.json",
    "calibration_A.json",
    "calibration_B.json",
    "ideal.json",
    "circuit_meta.json",
]

for f in required_files:
    if not os.path.exists(f):
        raise FileNotFoundError(f"Missing required file: {f}")

# ============================================================
# LOAD FROZEN SAVED FILES
# ============================================================

In [ ]:
with open("circuit_meta.json", "r") as f:
    meta_list = json.load(f)

with open("ideal.json", "r") as f:
    ideal = json.load(f)

with open("noisy_A.json", "r") as f:
    noisy_A = json.load(f)

with open("noisy_B.json", "r") as f:
    noisy_B = json.load(f)

with open("calibration_A.json", "r") as f:
    calibration_A = json.load(f)

with open("calibration_B.json", "r") as f:
    calibration_B = json.load(f)

meta_dict = {m["circuit_id"]: m for m in meta_list}

# Reconstruct lightweight circuits list from metadata only.
# qc is None because we do not need actual QuantumCircuit objects here.
circuits = [
    (m["circuit_id"], m["type"], None, m["n_qubits"], m["depth"])
    for m in meta_list
]

meta_first = [m["circuit_id"] for m in meta_list[:3]]
circ_first = [c[0] for c in circuits[:3]]

print("Meta first 3 IDs   :", meta_first)
print("Circuit first 3 IDs:", circ_first)

if meta_first != circ_first:
    raise ValueError("Reconstructed circuits order does not match circuit_meta.json")

print("✅ Loaded frozen saved artifacts only (offline mode)")


Meta first 3 IDs   : ['679b418c-f535-45da-8667-cf4baeee273b', 'd93990c9-1994-4c97-8db7-9500a6768871', '0b8feebf-2d4f-4aad-a729-e8d22a9fe0fc']
Circuit first 3 IDs: ['679b418c-f535-45da-8667-cf4baeee273b', 'd93990c9-1994-4c97-8db7-9500a6768871', '0b8feebf-2d4f-4aad-a729-e8d22a9fe0fc']
✅ Loaded frozen saved artifacts only (offline mode)


# ============================================================
# HELPERS
# ============================================================

In [ ]:
def mean_valid(lst):
    vals = [x for x in lst if x is not None and x > 0]
    return float(np.mean(vals)) if vals else 0.0

def mean_gate_error(calib):
    vals = list(calib.get("gate_errors", {}).values())
    vals = [v for v in vals if v is not None]
    return float(np.mean(vals)) if vals else 0.0

def dist_to_vec(dist, n_qubits):
    vec = np.zeros(MAX_STATES, dtype=np.float32)

    for bitstring, prob in dist.items():
        clean = str(bitstring).replace(" ", "").replace("_", "")
        if clean.startswith("0b"):
            clean = clean[2:]

        if not all(ch in "01" for ch in clean):
            print(f"Skipping malformed bitstring: {bitstring}")
            continue

        padded = clean.zfill(n_qubits)
        idx = int(padded, 2)

        if idx < MAX_STATES:
            vec[idx] = float(prob)

    s = vec.sum()
    if s > 0:
        vec /= s

    return vec

def extract_calibration_features(calib):
    return {
        "T1_mean": mean_valid(calib["T1"]),
        "T2_mean": mean_valid(calib["T2"]),
        "readout_error_mean": mean_valid(calib["readout_error"]),
        "cx_error_mean": mean_gate_error(calib),
    }

def normalize_np(p, eps=1e-8):
    p = np.asarray(p, dtype=np.float64)
    s = p.sum()
    if s <= 0:
        return np.ones_like(p) / len(p)
    return p / (s + eps)

def kl_divergence_np(p_true, p_pred, eps=1e-8):
    p_true = normalize_np(p_true, eps)
    p_pred = normalize_np(p_pred, eps)
    return float(np.sum(p_true * np.log((p_true + eps) / (p_pred + eps))))

def total_variation_np(p_true, p_pred):
    p_true = normalize_np(p_true)
    p_pred = normalize_np(p_pred)
    return float(0.5 * np.sum(np.abs(p_true - p_pred)))

# ============================================================
# BUILD ML-READY DATASET
# ============================================================

In [ ]:
calib_features_A = extract_calibration_features(calibration_A)
calib_features_B = extract_calibration_features(calibration_B)

dataset = []
skipped = 0

for backend_name, noisy_data, calib_features in [
    (calibration_A["backend"], noisy_A, calib_features_A),
    (calibration_B["backend"], noisy_B, calib_features_B),
]:
    for cid, noisy_entry in noisy_data.items():
        if cid not in ideal or cid not in meta_dict:
            skipped += 1
            continue

        m = meta_dict[cid]
        n = m["n_qubits"]
        gc = m["gate_counts"]

        scalar_features = np.array([
            float(n),
            float(m["depth"]),
            float(gc.get("cx", 0)),
            float(gc.get("h", 0)),
            float(gc.get("x", 0)),
            float(calib_features["T1_mean"]),
            float(calib_features["T2_mean"]),
            float(calib_features["readout_error_mean"]),
            float(calib_features["cx_error_mean"]),
        ], dtype=np.float32)

        noisy_dist = dist_to_vec(noisy_entry["distribution"], n)
        ideal_dist = dist_to_vec(ideal[cid], n)

        x = np.concatenate([scalar_features, noisy_dist]).astype(np.float32)
        y = ideal_dist.astype(np.float32)

        dataset.append({
            "circuit_id": cid,
            "backend": backend_name,
            "circuit_type": m["type"],
            "n_qubits": n,
            "x": x.tolist(),
            "y": y.tolist(),
            "noisy_dist": noisy_entry["distribution"],
            "ideal_dist": ideal[cid],
            "calibration": calib_features,
        })

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

from collections import Counter
backend_counts = Counter(d["backend"] for d in dataset)
type_counts = Counter(d["circuit_type"] for d in dataset)

print("\n✅ Dataset built:", len(dataset), "total samples", f"({skipped} skipped)")
print("\nBy backend:")
for b, c in backend_counts.items():
    print(f"  {b}: {c} samples")
print("\nBy circuit type:")
for t, c in type_counts.items():
    print(f"  {t}: {c} samples")
print(f"\nFeature vector shape: x={len(dataset[0]['x'])}, y={len(dataset[0]['y'])}")


✅ Dataset built: 170 total samples (0 skipped)

By backend:
  ibm_fez: 85 samples
  ibm_marrakesh: 85 samples

By circuit type:
  random: 80 samples
  bell: 30 samples
  ghz: 30 samples
  qft: 30 samples

Feature vector shape: x=41, y=32


# ============================================================
# STANDARDIZE FIRST 9 FEATURES USING BACKEND A ONLY
# ============================================================

In [ ]:
BACKEND_A = calibration_A["backend"]
BACKEND_B = calibration_B["backend"]

a_scalars = np.array(
    [d["x"][:SCALAR_DIM] for d in dataset if d["backend"] == BACKEND_A],
    dtype=np.float32
)

if len(a_scalars) == 0:
    raise ValueError(f"No samples found for backend {BACKEND_A}")

mean_9 = a_scalars.mean(axis=0)
std_9 = a_scalars.std(axis=0) + 1e-8

dataset_std = copy.deepcopy(dataset)
for d in dataset_std:
    x = np.array(d["x"], dtype=np.float32)
    x[:SCALAR_DIM] = (x[:SCALAR_DIM] - mean_9) / std_9
    d["x"] = x.tolist()

with open("dataset_standardized.json", "w") as f:
    json.dump(dataset_std, f, indent=2)

print("✅ Saved dataset_standardized.json")


✅ Saved dataset_standardized.json


# ============================================================
# TRAIN / TEST SPLIT
# ============================================================

In [ ]:
train_data = [d for d in dataset_std if d["backend"] == BACKEND_A]
test_data  = [d for d in dataset_std if d["backend"] == BACKEND_B]

print(f"Training samples (Backend A): {len(train_data)}")
print(f"Test samples (Backend B):     {len(test_data)}")

if len(train_data) == 0 or len(test_data) == 0:
    raise ValueError("Need both Backend A and Backend B samples.")

Training samples (Backend A): 85
Test samples (Backend B):     85


# ============================================================
# MODEL
# ============================================================

In [ ]:
class ResidualNoiseAdapter(nn.Module):
    def __init__(self, input_dim=41, output_dim=32):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(0.10),

            nn.Linear(128, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(0.10),

            nn.Linear(128, 64),
            nn.LayerNorm(64),
            nn.GELU()
        )
        self.head = nn.Linear(64, output_dim)

    def forward(self, x):
        noisy_part = x[:, 9:]
        correction = self.head(self.backbone(x))
        logits = noisy_part + correction
        return logits

def kl_loss_from_logits(logits, target_probs, eps=1e-8):
    target_probs = target_probs / (target_probs.sum(dim=1, keepdim=True) + eps)
    pred_log_probs = F.log_softmax(logits, dim=1)
    return F.kl_div(pred_log_probs, target_probs, reduction="batchmean")

def predict_probs(model, x_tensor):
    model.eval()
    with torch.no_grad():
        logits = model(x_tensor)
        probs = torch.softmax(logits, dim=1)
    return probs

# ============================================================
# TRAINING DATA
# ============================================================

In [ ]:
X_train_full = torch.tensor([d["x"] for d in train_data], dtype=torch.float32)
Y_train_full = torch.tensor([d["y"] for d in train_data], dtype=torch.float32)

full_ds = TensorDataset(X_train_full, Y_train_full)

n_total = len(full_ds)
n_val = max(10, int(0.20 * n_total))
n_train = n_total - n_val

train_ds, val_ds = random_split(
    full_ds,
    [n_train, n_val],
    generator=torch.Generator().manual_seed(SEED)
)

# Deterministic shuffle
train_gen = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, generator=train_gen)
val_loader   = DataLoader(val_ds, batch_size=32, shuffle=False)

# ============================================================
# TRAIN MODEL ON BACKEND A
# ============================================================

In [ ]:
model = ResidualNoiseAdapter(input_dim=INPUT_DIM, output_dim=OUTPUT_DIM).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=12
)

best_state = None
best_val = float("inf")
best_epoch = 0
bad_epochs = 0
patience = 25
num_epochs = 250
stopped_epoch = num_epochs - 1

train_history = []
val_history = []
lr_history = []

print("\nTraining improved model on Backend A...")
print(f"{'Epoch':<8} {'Train Loss':<15} {'Val Loss':<15} {'LR':<12}")
print("-" * 55)

for epoch in range(num_epochs):
    model.train()
    train_loss_sum = 0.0

    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()
        logits = model(xb)
        loss = kl_loss_from_logits(logits, yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        train_loss_sum += loss.item() * xb.size(0)

    train_loss = train_loss_sum / len(train_loader.dataset)

    model.eval()
    val_loss_sum = 0.0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = kl_loss_from_logits(logits, yb)
            val_loss_sum += loss.item() * xb.size(0)

    val_loss = val_loss_sum / len(val_loader.dataset)
    scheduler.step(val_loss)

    train_history.append(train_loss)
    val_history.append(val_loss)
    lr_history.append(optimizer.param_groups[0]["lr"])

    if epoch % 20 == 0 or epoch == num_epochs - 1:
        lr_now = optimizer.param_groups[0]["lr"]
        print(f"{epoch:<8d} {train_loss:<15.6f} {val_loss:<15.6f} {lr_now:<12.6f}")

    if val_loss < best_val:
        best_val = val_loss
        best_epoch = epoch
        best_state = copy.deepcopy(model.state_dict())
        bad_epochs = 0
    else:
        bad_epochs += 1

    if bad_epochs >= patience:
        stopped_epoch = epoch
        print(f"Early stopping at epoch {epoch}")
        break

model.load_state_dict(best_state)
torch.save(model.state_dict(), "model_backend_a_improved.pt")
print(f"\n✅ Saved best model to model_backend_a_improved.pt")
print(f"Best validation loss: {best_val:.6f} (epoch {best_epoch})")

training_history_data = {
    "epochs": list(range(1, len(train_history) + 1)),
    "train_loss": train_history,
    "val_loss": val_history,
    "lr": lr_history,
    "best_epoch": best_epoch,
    "best_val_loss": best_val,
    "stopped_epoch": stopped_epoch,
}
with open("training_history.json", "w") as f:
    json.dump(training_history_data, f, indent=2)
print("✅ Saved training_history.json")


Training improved model on Backend A...
Epoch    Train Loss      Val Loss        LR          
-------------------------------------------------------
0        2.026577        1.588072        0.001000    
20       0.175080        0.494422        0.001000    
40       0.066004        0.513518        0.000500    
Early stopping at epoch 52

✅ Saved best model to model_backend_a_improved.pt
Best validation loss: 0.460505 (epoch 27)
✅ Saved training_history.json


# ============================================================
# EVALUATION HELPERS
# ============================================================

In [ ]:
def evaluate_model(model, dataset_list, device="cpu", return_per_sample=False):
    if len(dataset_list) == 0:
        return {"kl": None, "tv": None, "predictions": [], "per_sample": []}

    X = torch.tensor([d["x"] for d in dataset_list], dtype=torch.float32).to(device)
    Y = np.array([d["y"] for d in dataset_list], dtype=np.float64)

    probs = predict_probs(model, X).cpu().numpy()

    kls = []
    tvs = []
    preds = []
    per_sample = []

    for i, (pred, true) in enumerate(zip(probs, Y)):
        kl_val = kl_divergence_np(true, pred)
        tv_val = total_variation_np(true, pred)
        kls.append(kl_val)
        tvs.append(tv_val)
        preds.append(pred.tolist())

        if return_per_sample:
            d = dataset_list[i]
            noisy_vec = np.array(d["x"][SCALAR_DIM:], dtype=np.float64)
            kl_noisy = kl_divergence_np(true, noisy_vec)
            tv_noisy = total_variation_np(true, noisy_vec)
            per_sample.append({
                "circuit_id": d["circuit_id"],
                "backend": d["backend"],
                "circuit_type": d["circuit_type"],
                "n_qubits": d["n_qubits"],
                "ideal_dist": true.tolist(),
                "noisy_dist": noisy_vec.tolist(),
                "predicted_dist": pred.tolist(),
                "kl_pred": kl_val,
                "tv_pred": tv_val,
                "kl_noisy": kl_noisy,
                "tv_noisy": tv_noisy,
            })

    result = {
        "kl": float(np.mean(kls)),
        "tv": float(np.mean(tvs)),
        "predictions": preds,
    }
    if return_per_sample:
        result["per_sample"] = per_sample
    return result

def evaluate_in_domain_split(model, source_data, seed=42):
    rng = np.random.RandomState(seed)
    idx = np.arange(len(source_data))
    rng.shuffle(idx)

    split = int(0.8 * len(source_data))
    eval_idx = idx[split:]
    eval_data = [source_data[i] for i in eval_idx]

    return evaluate_model(model, eval_data, device=device)


# ============================================================
# ZERO-SHOT ON BACKEND B
# ============================================================


In [ ]:
print("\n" + "=" * 65)
print("EXPERIMENT: Few-Shot Cross-Device Adaptation")
print(f"Train: {BACKEND_A} | Test: {BACKEND_B}")
print("=" * 65)

zero_shot = evaluate_model(model, test_data, device=device, return_per_sample=True)

print("\nCondition 1 — Zero-shot (no adaptation):")
print(f"  No adaptation                       KL: {zero_shot['kl']:.4f}  |  TV: {zero_shot['tv']:.4f}")

# Real Figure 5 artifact
per_sample = zero_shot["per_sample"]
best_example_idx = max(range(len(per_sample)), key=lambda i: per_sample[i]["kl_noisy"])
example_sample = per_sample[best_example_idx]

example_prediction = {
    "circuit_id":     example_sample["circuit_id"],
    "backend":        example_sample["backend"],
    "circuit_type":   example_sample["circuit_type"],
    "n_qubits":       example_sample["n_qubits"],
    "ideal_dist":     example_sample["ideal_dist"],
    "noisy_dist":     example_sample["noisy_dist"],
    "predicted_dist": example_sample["predicted_dist"],
    "kl_pred":        example_sample["kl_pred"],
    "tv_pred":        example_sample["tv_pred"],
    "kl_noisy":       example_sample["kl_noisy"],
    "tv_noisy":       example_sample["tv_noisy"],
    "selection_rule": "highest KL(ideal || noisy) among all Backend B test samples",
}

with open("example_prediction.json", "w") as f:
    json.dump(example_prediction, f, indent=2)
print("✅ Saved example_prediction.json")



EXPERIMENT: Few-Shot Cross-Device Adaptation
Train: ibm_fez | Test: ibm_marrakesh

Condition 1 — Zero-shot (no adaptation):
  No adaptation                       KL: 1.6706  |  TV: 0.5282
✅ Saved example_prediction.json


# ============================================================
# FEW-SHOT ADAPTATION
# ============================================================

In [ ]:
def freeze_all(model):
    for p in model.parameters():
        p.requires_grad = False

def unfreeze_head_only(model):
    freeze_all(model)
    for p in model.head.parameters():
        p.requires_grad = True

def unfreeze_last_block_and_head(model):
    freeze_all(model)
    for idx in [8, 9]:
        for p in model.backbone[idx].parameters():
            p.requires_grad = True
    for p in model.head.parameters():
        p.requires_grad = True

def adapt_few_shot(base_model, source_train_data, target_data, K, seed=42, device="cpu"):
    rng = np.random.RandomState(seed)
    indices = np.arange(len(target_data))
    rng.shuffle(indices)

    few_idx = indices[:K]
    eval_idx = indices[K:]

    few_shot = [target_data[i] for i in few_idx]
    target_eval = [target_data[i] for i in eval_idx]

    replay_size = min(24, len(source_train_data))
    replay_idx = rng.choice(len(source_train_data), size=replay_size, replace=False)
    replay = [source_train_data[i] for i in replay_idx]

    adapt_data = few_shot + replay

    X_adapt = torch.tensor([d["x"] for d in adapt_data], dtype=torch.float32)
    Y_adapt = torch.tensor([d["y"] for d in adapt_data], dtype=torch.float32)

    adapt_gen = torch.Generator().manual_seed(seed)
    adapt_loader = DataLoader(
        TensorDataset(X_adapt, Y_adapt),
        batch_size=min(16, len(adapt_data)),
        shuffle=True,
        generator=adapt_gen
    )

    X_few = torch.tensor([d["x"] for d in few_shot], dtype=torch.float32).to(device)
    Y_few = torch.tensor([d["y"] for d in few_shot], dtype=torch.float32).to(device)

    model_adapt = copy.deepcopy(base_model).to(device)

    if K <= 10:
        unfreeze_head_only(model_adapt)
        lr = 1e-4
        epochs = 60
    else:
        unfreeze_last_block_and_head(model_adapt)
        lr = 5e-5
        epochs = 80

    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model_adapt.parameters()),
        lr=lr,
        weight_decay=1e-4
    )

    best_state = copy.deepcopy(model_adapt.state_dict())
    best_score = float("inf")
    patience = 12
    bad_epochs = 0

    for epoch in range(epochs):
        model_adapt.train()

        for xb, yb in adapt_loader:
            xb, yb = xb.to(device), yb.to(device)

            optimizer.zero_grad()
            logits = model_adapt(xb)
            loss = kl_loss_from_logits(logits, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model_adapt.parameters(), 1.0)
            optimizer.step()

        model_adapt.eval()
        with torch.no_grad():
            few_logits = model_adapt(X_few)
            few_loss = kl_loss_from_logits(few_logits, Y_few).item()

        if few_loss < best_score:
            best_score = few_loss
            best_state = copy.deepcopy(model_adapt.state_dict())
            bad_epochs = 0
        else:
            bad_epochs += 1

        if bad_epochs >= patience:
            break

    model_adapt.load_state_dict(best_state)

    eval_result = evaluate_model(model_adapt, target_eval, device=device)
    return {
        "K": K,
        "few_shot_size": len(few_shot),
        "eval_size": len(target_eval),
        "kl": eval_result["kl"],
        "tv": eval_result["tv"],
    }

print("\nCondition 2 — Few-shot fine-tuning:")
few_shot_results = {}

for K in [5, 10, 20]:
    kls = []
    tvs = []

    for seed in [0, 1, 2, 3, 4]:
        out = adapt_few_shot(
            base_model=model,
            source_train_data=train_data,
            target_data=test_data,
            K=K,
            seed=seed,
            device=device
        )
        kls.append(out["kl"])
        tvs.append(out["tv"])

    mean_kl = float(np.mean(kls))
    std_kl = float(np.std(kls))
    mean_tv = float(np.mean(tvs))
    std_tv = float(np.std(tvs))

    few_shot_results[K] = {
        "kl_mean": mean_kl,
        "kl_std": std_kl,
        "tv_mean": mean_tv,
        "tv_std": std_tv,
    }

    print(f"  K={K:<4d} KL: {mean_kl:.4f} ± {std_kl:.4f}  |  TV: {mean_tv:.4f} ± {std_tv:.4f}")



Condition 2 — Few-shot fine-tuning:
  K=5    KL: 1.5874 ± 0.0849  |  TV: 0.5287 ± 0.0081
  K=10   KL: 1.5172 ± 0.0669  |  TV: 0.5290 ± 0.0088
  K=20   KL: 1.1924 ± 0.0630  |  TV: 0.5503 ± 0.0095


# ============================================================
# IMPROVEMENT SUMMARY
# ============================================================

In [ ]:
print("\n✅ Results summary")
print("\n📊 KL Improvement over no-adaptation baseline:")
for K in [5, 10, 20]:
    base = zero_shot["kl"]
    new = few_shot_results[K]["kl_mean"]
    improvement_pct = 100.0 * (base - new) / base
    print(f"   K={K}: {improvement_pct:+.1f}% improvement")


✅ Results summary

📊 KL Improvement over no-adaptation baseline:
   K=5: +5.0% improvement
   K=10: +9.2% improvement
   K=20: +28.6% improvement


# ============================================================
# DRIFT SUMMARY
# ============================================================

In [ ]:
A_T1 = mean_valid(calibration_A["T1"]) * 1e6
A_T2 = mean_valid(calibration_A["T2"]) * 1e6
A_RO = mean_valid(calibration_A["readout_error"])
A_CX = mean_gate_error(calibration_A)

B_T1 = mean_valid(calibration_B["T1"]) * 1e6
B_T2 = mean_valid(calibration_B["T2"]) * 1e6
B_RO = mean_valid(calibration_B["readout_error"])
B_CX = mean_gate_error(calibration_B)

print("\n📊 Calibration Comparison (Frozen Saved Files)")
print("=" * 55)
print(f"{'Property':<24} {'Backend A':>10} {'Backend B':>10} {'Δ':>10}")
print("-" * 55)
print(f"{'T1 (µs)':<24} {A_T1:>10.1f} {B_T1:>10.1f} {B_T1 - A_T1:>+10.1f}")
print(f"{'T2 (µs)':<24} {A_T2:>10.1f} {B_T2:>10.1f} {B_T2 - A_T2:>+10.1f}")
print(f"{'Readout error':<24} {A_RO:>10.4f} {B_RO:>10.4f} {B_RO - A_RO:>+10.4f}")
print(f"{'CX gate error':<24} {A_CX:>10.4f} {B_CX:>10.4f} {B_CX - A_CX:>+10.4f}")
print("=" * 55)

in_domain = evaluate_in_domain_split(model, train_data, seed=SEED)

print("\n📊 Performance Degradation from Drift:")
print(f"   Model on Backend A (in-domain):  KL = {in_domain['kl']:.4f}")
print(f"   Model on Backend B (no adapt):   KL = {zero_shot['kl']:.4f}")
print(f"   Model on Backend B (K=20 shots): KL = {few_shot_results[20]['kl_mean']:.4f}")



📊 Calibration Comparison (Frozen Saved Files)
Property                  Backend A  Backend B          Δ
-------------------------------------------------------
T1 (µs)                       142.4      192.8      +50.5
T2 (µs)                       104.1      114.0      +10.0
Readout error                0.0285     0.0335    +0.0050
CX gate error                0.0328     0.0560    +0.0232

📊 Performance Degradation from Drift:
   Model on Backend A (in-domain):  KL = 0.3014
   Model on Backend B (no adapt):   KL = 1.6706
   Model on Backend B (K=20 shots): KL = 1.1924


# ============================================================
# FEATURE ABLATION
# ============================================================

In [ ]:
# ============================================================
# FEATURE ABLATION
# ============================================================

ablation_map = {
    "Remove T1": 5,
    "Remove T2": 6,
    "Remove Readout Error": 7,
    "Remove CX Gate Error": 8,
}

def ablate_dataset(dataset_list, index_to_zero):
    out = []
    for d in dataset_list:
        d2 = copy.deepcopy(d)
        x = np.array(d2["x"], dtype=np.float32)
        x[index_to_zero] = 0.0
        d2["x"] = x.tolist()
        out.append(d2)
    return out

ablation_results = {}
ablation_deltas = {}

baseline_kl = zero_shot["kl"]
ablation_results["All features (baseline)"] = baseline_kl

# Threshold for "meaningful" effect
EPS = 1e-3

print("\n📊 Ablation Study — Calibration Feature Effects on Zero-Shot Transfer")
print("=" * 76)
print(f"{'Ablation':<30} {'KL':>10} {'Δ vs Baseline':>18} {'Interpretation':>20}")
print("-" * 76)
print(f"{'All features (baseline)':<30} {baseline_kl:>10.4f} {'—':>18} {'baseline':>20}")

for name, idx_zero in ablation_map.items():
    ablated_test = ablate_dataset(test_data, idx_zero)
    result = evaluate_model(model, ablated_test, device=device)

    delta = result["kl"] - baseline_kl
    ablation_results[name] = result["kl"]
    ablation_deltas[name] = delta

    if delta > EPS:
        interp = "useful feature"
    elif delta < -EPS:
        interp = "mismatch feature"
    else:
        interp = "no significant effect"

    print(f"{name:<30} {result['kl']:>10.4f} {delta:>+18.4f} {interp:>20}")

# Feature whose removal most HELPS zero-shot transfer
largest_decrease_feature = None
largest_decrease = 0.0

# Feature whose removal most HURTS zero-shot transfer
largest_increase_feature = None
largest_increase = 0.0

for name in ablation_map:
    delta = ablation_deltas[name]

    # Most harmful / mismatched feature: removal decreases KL the most
    if delta < -EPS:
        decrease_amount = -delta
        if decrease_amount > largest_decrease:
            largest_decrease = decrease_amount
            largest_decrease_feature = name

    # Most useful feature: removal increases KL the most
    if delta > EPS:
        increase_amount = delta
        if increase_amount > largest_increase:
            largest_increase = increase_amount
            largest_increase_feature = name

print("=" * 76)

if largest_decrease_feature is not None:
    print(f"\n🔑 Strongest mismatch feature: '{largest_decrease_feature}'")
    print(f"   Removing it reduces KL by {largest_decrease:.4f},")
    print("   indicating that this feature introduces significant cross-device distribution mismatch.")
else:
    print("\n🔑 No feature showed a meaningful beneficial effect when removed.")

if largest_increase_feature is not None:
    print(f"\n🔑 Most useful feature: '{largest_increase_feature}'")
    print(f"   Removing it increases KL by {largest_increase:.4f},")
    print("   indicating that this feature contributes positively to transfer.")
else:
    print("\n🔑 No feature showed a meaningful positive contribution under zero-shot transfer.")

# Save a compact summary for later use in the paper / figures
ablation_summary = {
    "baseline_kl": baseline_kl,
    "ablation_results": ablation_results,
    "ablation_deltas": ablation_deltas,
    "significance_threshold": EPS,
    "strongest_mismatch_feature": largest_decrease_feature,
    "strongest_mismatch_delta": largest_decrease,
    "most_useful_feature": largest_increase_feature,
    "most_useful_delta": largest_increase,
}


📊 Ablation Study — Calibration Feature Effects on Zero-Shot Transfer
Ablation                               KL      Δ vs Baseline       Interpretation
----------------------------------------------------------------------------
All features (baseline)            1.6706                  —             baseline
Remove T1                          1.6708            +0.0001 no significant effect
Remove T2                          1.6705            -0.0002 no significant effect
Remove Readout Error               1.6151            -0.0556     mismatch feature
Remove CX Gate Error               1.3771            -0.2935     mismatch feature

🔑 Strongest mismatch feature: 'Remove CX Gate Error'
   Removing it reduces KL by 0.2935,
   indicating that this feature introduces significant cross-device distribution mismatch.

🔑 No feature showed a meaningful positive contribution under zero-shot transfer.


# ============================================================
# SAVE RESULTS
# ============================================================

In [ ]:
# ============================================================
# SAVE RESULTS (FINAL CORRECT VERSION)
# ============================================================

experiment_results = {
    "backend_a": BACKEND_A,
    "backend_b": BACKEND_B,
    "dataset_size": len(dataset_std),
    "train_size": len(train_data),
    "test_size": len(test_data),

    # Training info
    "best_val_loss": best_val,
    "best_epoch": best_epoch,          # zero-based
    "stopped_epoch": stopped_epoch,    # zero-based

    # Performance
    "in_domain": {
        "kl": in_domain["kl"],
        "tv": in_domain["tv"],
    },
    "zero_shot": {
        "kl": zero_shot["kl"],
        "tv": zero_shot["tv"],
    },
    "few_shot": few_shot_results,

    # Drift
    "drift_summary": {
        "A_T1": A_T1,
        "A_T2": A_T2,
        "A_readout_error": A_RO,
        "A_cx_error": A_CX,
        "B_T1": B_T1,
        "B_T2": B_T2,
        "B_readout_error": B_RO,
        "B_cx_error": B_CX,
    }
}

with open("experiment_results_fixed.json", "w") as f:
    json.dump(experiment_results, f, indent=2)


# ============================================================
# SAVE ABLATION (CORRECT + EXPLICIT)
# ============================================================

ablation_save = {
    "baseline_kl": baseline_kl,
    "ablation_results": ablation_results,
    "ablation_deltas": ablation_deltas,

    # Correct interpretation fields
    "strongest_mismatch_feature": largest_decrease_feature,
    "strongest_mismatch_delta": largest_decrease,

    "most_useful_feature": largest_increase_feature if largest_increase > 0 else None,
    "most_useful_delta": largest_increase if largest_increase > 0 else 0.0,
}

with open("ablation_results_fixed.json", "w") as f:
    json.dump(ablation_save, f, indent=2)


# ============================================================
# FINAL LOG
# ============================================================

print("\n✅ Saved experiment_results_fixed.json")
print("✅ Saved ablation_results_fixed.json")
print("✅ Saved training_history.json")
print("✅ Saved example_prediction.json")
print("✅ Done")


✅ Saved experiment_results_fixed.json
✅ Saved ablation_results_fixed.json
✅ Saved training_history.json
✅ Saved example_prediction.json
✅ Done


# ============================================================
# QUANTUM NOISE FEW-SHOT TRANSFER — PAPER-READY VISUALIZATION
# Colab-Ready | Publication Quality | Verified Numbers
# ============================================================
#
# REQUIRED FILES (upload to Colab /content/ before running):
#   experiment_results_fixed.json
#   ablation_results_fixed.json
#   training_history.json        (saved by the reproduce notebook)
#   example_prediction.json      (saved by the reproduce notebook)
#
# OUTPUTS  →  paper_viz/
#   dashboard.png / .pdf         ← main submission figure
#   fig1_kl_transfer.png
#   fig2_tv_transfer.png
#   fig3_ablation.png
#   fig4_training.png
#   fig5_example_prediction.png
#   fig6_results_table.png
#   fig7_dataset_overview.png    (needs dataset.json)
#   fig8_calibration_drift.png
#   supplementary_summary.png
#   all_figures.pdf
# ============================================================

In [ ]:
# ── Cell 2 ── Imports ─────────────────────────────────────────
import os, json, warnings
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch, Rectangle
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.ticker import MaxNLocator
warnings.filterwarnings("ignore")

# ── Cell 3 ── Output directory ────────────────────────────────
OUTDIR = "paper_viz"
os.makedirs(OUTDIR, exist_ok=True)

# ── Cell 4 ── Colour palette & rcParams ──────────────────────
C_BG     = "#FAFAFA"
C_PANEL  = "#FFFFFF"
C_GRID   = "#E8ECF0"
C_TEXT   = "#1A202C"
C_SUB    = "#4A5568"
C_BORDER = "#CBD5E0"
C_SLATE  = "#2D3748"
C_LIGHT  = "#EDF2F7"

C_BLUE   = "#2B6CB0"
C_BLUE_L = "#BEE3F8"
C_RED    = "#C53030"
C_GREEN  = "#276749"
C_ORANGE = "#C05621"
C_PURPLE = "#553C9A"
C_TEAL   = "#2C7A7B"
C_AMBER  = "#B7791F"
PALETTE  = [C_BLUE, C_ORANGE, C_GREEN, C_PURPLE, C_TEAL, C_AMBER]

matplotlib.rcParams.update({
    "figure.facecolor"    : C_BG,
    "figure.dpi"          : 150,
    "axes.facecolor"      : C_PANEL,
    "axes.edgecolor"      : C_BORDER,
    "axes.linewidth"      : 0.8,
    "axes.labelcolor"     : C_TEXT,
    "axes.labelsize"      : 10,
    "axes.titlesize"      : 11,
    "axes.titleweight"    : "bold",
    "axes.titlepad"       : 8,
    "axes.spines.top"     : False,
    "axes.spines.right"   : False,
    "axes.grid"           : True,
    "axes.axisbelow"      : True,
    "grid.color"          : C_GRID,
    "grid.linewidth"      : 0.6,
    "grid.linestyle"      : "-",
    "grid.alpha"          : 1.0,
    "xtick.color"         : C_TEXT,
    "ytick.color"         : C_TEXT,
    "xtick.labelsize"     : 9,
    "ytick.labelsize"     : 9,
    "xtick.major.size"    : 3,
    "ytick.major.size"    : 3,
    "legend.frameon"      : True,
    "legend.facecolor"    : C_PANEL,
    "legend.edgecolor"    : C_BORDER,
    "legend.fontsize"     : 8.5,
    "legend.framealpha"   : 0.92,
    "font.family"         : "DejaVu Sans",
    "font.size"           : 10,
    "lines.linewidth"     : 1.8,
    "lines.markersize"    : 6,
    "errorbar.capsize"    : 4,
    "savefig.facecolor"   : C_BG,
    "savefig.dpi"         : 300,
    "savefig.bbox"        : "tight",
    "savefig.pad_inches"  : 0.15,
})

# ── Cell 5 ── Helpers ─────────────────────────────────────────
def savefig(fig, name, dpi=300):
    path = os.path.join(OUTDIR, name)
    fig.savefig(path, dpi=dpi)
    print(f"  ✓  {path}")
    return path

def add_bar_values(ax, fmt="{:.4f}", fontsize=8.5, color=None):
    ymax = ax.get_ylim()[1]
    pad  = ymax * 0.015
    for p in ax.patches:
        h = p.get_height()
        if h == 0: continue
        ax.text(p.get_x() + p.get_width()/2, h + pad,
                fmt.format(h), ha="center", va="bottom",
                fontsize=fontsize, color=color or C_SUB)

def style_ax(ax, title=None, xlabel=None, ylabel=None, grid_axis="y"):
    if title:   ax.set_title(title)
    if xlabel:  ax.set_xlabel(xlabel, labelpad=4)
    if ylabel:  ax.set_ylabel(ylabel, labelpad=4)
    ax.grid(True, axis=grid_axis, linewidth=0.6, color=C_GRID)
    ax.set_axisbelow(True)

def draw_stat_card(ax, label, value_str, sub="", accent=C_BLUE, value_size=20):
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_axis_off()
    ax.add_patch(FancyBboxPatch((0,0), 1, 1,
        boxstyle="round,pad=0.03,rounding_size=0.06",
        linewidth=0.8, edgecolor=C_BORDER, facecolor=C_PANEL,
        transform=ax.transAxes, clip_on=False))
    ax.add_patch(FancyBboxPatch((0, 0.88), 1, 0.12,
        boxstyle="round,pad=0.0,rounding_size=0.06",
        linewidth=0, facecolor=accent, alpha=0.92,
        transform=ax.transAxes, clip_on=False, zorder=4))
    ax.text(0.08, 0.71, label, transform=ax.transAxes,
            fontsize=9, fontweight="bold", color=C_SUB, va="center")
    ax.text(0.08, 0.44, value_str, transform=ax.transAxes,
            fontsize=value_size, fontweight="bold", color=C_TEXT, va="center")
    if sub:
        ax.text(0.08, 0.16, sub, transform=ax.transAxes,
                fontsize=8, color=C_SUB, va="center")

# ── Cell 6 ── Load all result files ──────────────────────────
required = [
    "experiment_results_fixed.json",
    "ablation_results_fixed.json",
    "training_history.json",
    "example_prediction.json",
]
for f in required:
    if not os.path.exists(f):
        raise FileNotFoundError(
            f"❌  Missing: {f}\n"
            f"   Upload it to Colab (/content/) before running."
        )

with open("experiment_results_fixed.json") as f: exp = json.load(f)
with open("ablation_results_fixed.json")   as f: abl = json.load(f)
with open("training_history.json")         as f: th  = json.load(f)
with open("example_prediction.json")       as f: ex  = json.load(f)

# Optional dataset.json for Fig 7
has_dataset = os.path.exists("dataset.json")
if has_dataset:
    with open("dataset.json") as f: dataset = json.load(f)

# ── Cell 7 ── Unpack verified numbers ────────────────────────
BACKEND_A  = exp["backend_a"]          # ibm_fez
BACKEND_B  = exp["backend_b"]          # ibm_marrakesh

IN_KL      = exp["in_domain"]["kl"]    # 0.3014
ZERO_KL    = exp["zero_shot"]["kl"]    # 1.6706
ZERO_TV    = exp["zero_shot"]["tv"]    # 0.5282

few        = exp["few_shot"]
K_VALS     = sorted(int(k) for k in few.keys())   # [5, 10, 20]

def get_fs(k, key):
    return few[str(k)][key] if str(k) in few else few[k][key]

KL_MEANS   = [get_fs(k, "kl_mean") for k in K_VALS]
KL_STDS    = [get_fs(k, "kl_std")  for k in K_VALS]
TV_MEANS   = [get_fs(k, "tv_mean") for k in K_VALS]
TV_STDS    = [get_fs(k, "tv_std")  for k in K_VALS]

# Improvement metrics (computed, not taken from JSON)
IMPROV_REL = [100.0*(ZERO_KL - km)/ZERO_KL for km in KL_MEANS]  # vs zero-shot
GAP        = ZERO_KL - IN_KL
IMPROV_GAP = [100.0*(ZERO_KL - km)/GAP for km in KL_MEANS]      # % gap recovery

BEST_K_IDX = int(np.argmin(KL_MEANS))
BEST_K     = K_VALS[BEST_K_IDX]
BEST_KL    = KL_MEANS[BEST_K_IDX]

# Drift
d = exp["drift_summary"]
A_T1, A_T2, A_RO, A_CX = d["A_T1"], d["A_T2"], d["A_readout_error"], d["A_cx_error"]
B_T1, B_T2, B_RO, B_CX = d["B_T1"], d["B_T2"], d["B_readout_error"], d["B_cx_error"]

# Ablation  (new interpretation: all deltas are negative → mismatch features)
baseline_kl = abl["baseline_kl"]       # = ZERO_KL = 1.6706
abl_results = abl["ablation_results"]
abl_deltas  = abl.get("ablation_deltas", {})

ABL_ORDER = [
    "All features (baseline)",
    "Remove T1",
    "Remove T2",
    "Remove Readout Error",
    "Remove CX Gate Error",
]
ABL_KEYS_MAP = {
    "All features (baseline)": "All features (baseline)",
    "Remove T1":               "Remove T1",
    "Remove T2":               "Remove T2",
    "Remove Readout Error":    "Remove Readout Error",
    "Remove CX Gate Error":    "Remove CX Gate Error",
}
ABL_VALS   = [abl_results[ABL_KEYS_MAP[k]] for k in ABL_ORDER]
ABL_DELTAS = [abl_deltas.get(ABL_KEYS_MAP[k], 0.0) if k != "All features (baseline)" else 0.0
              for k in ABL_ORDER]

# Training history
train_loss = th["train_loss"]
val_loss   = th["val_loss"]
best_epoch = th["best_epoch"]
best_val   = th["best_val_loss"]
stopped    = th["stopped_epoch"]

# Example prediction
EX_N       = ex["n_qubits"]
EX_TYPE    = ex["circuit_type"]
EX_IDEAL   = np.array(list(ex["ideal_dist"].values()) if isinstance(ex["ideal_dist"], dict)
                       else ex["ideal_dist"])
EX_NOISY   = np.array(list(ex["noisy_dist"].values()) if isinstance(ex["noisy_dist"], dict)
                       else ex["noisy_dist"])
EX_PRED    = np.array(list(ex["predicted_dist"].values()) if isinstance(ex["predicted_dist"], dict)
                       else ex["predicted_dist"])
EX_KL_PRED   = ex["kl_pred"]
EX_TV_PRED   = ex["tv_pred"]
EX_KL_NOISY  = ex["kl_noisy"]
EX_TV_NOISY  = ex["tv_noisy"]

def _pad(v, n):
    out = np.zeros(n, dtype=np.float64)
    out[:min(len(v), n)] = v[:min(len(v), n)]
    return out

N_STATES = 2 ** EX_N
EX_IDEAL = _pad(EX_IDEAL, N_STATES)
EX_NOISY = _pad(EX_NOISY, N_STATES)
EX_PRED  = _pad(EX_PRED,  N_STATES)
STATE_LABELS = [format(i, f"0{EX_N}b") for i in range(N_STATES)]

print(f"  ✓  Loaded all results")
print(f"     Backend A : {BACKEND_A}  |  Backend B : {BACKEND_B}")
print(f"     In-domain KL = {IN_KL:.4f}")
print(f"     Zero-shot KL = {ZERO_KL:.4f}  ({ZERO_KL/IN_KL:.1f}× degradation)")
print(f"     K=20 KL      = {KL_MEANS[-1]:.4f}  ({IMPROV_REL[-1]:.1f}% vs zero-shot, "
      f"{IMPROV_GAP[-1]:.1f}% gap recovery)")


# ============================================================
# ── FIGURE FUNCTIONS ─────────────────────────────────────────
# ============================================================

def fig_kl_transfer():
    """Fig 1 — KL divergence vs K with CI band, both reference lines, % annotations."""
    fig, ax = plt.subplots(figsize=(6.5, 4.6))
    k_x = np.array(K_VALS)

    ax.fill_between(k_x,
        np.array(KL_MEANS) - np.array(KL_STDS),
        np.array(KL_MEANS) + np.array(KL_STDS),
        color=C_BLUE_L, alpha=0.50)

    ax.axhline(ZERO_KL, color=C_RED, linestyle="--", linewidth=1.8,
               label=f"Zero-shot  {ZERO_KL:.4f}", zorder=2)
    ax.axhline(IN_KL, color=C_GREEN, linestyle=":", linewidth=1.6,
               label=f"In-domain  {IN_KL:.4f}", zorder=2)
    ax.errorbar(k_x, KL_MEANS, yerr=KL_STDS,
                fmt="o-", color=C_BLUE, linewidth=2.2, markersize=7, capsize=4,
                markerfacecolor="white", markeredgewidth=2,
                label="Few-shot adaptation", zorder=3)

    for k, km, ri, gi in zip(K_VALS, KL_MEANS, IMPROV_REL, IMPROV_GAP):
        ax.annotate(f"−{ri:.1f}%\n({gi:.1f}% gap)",
                    xy=(k, km), xytext=(0, 12), textcoords="offset points",
                    ha="center", fontsize=7.5, color=C_BLUE, fontweight="bold")

    style_ax(ax,
             title="Cross-Device Transfer — KL Divergence vs Few-Shot K",
             xlabel="Number of target-device samples  (K)",
             ylabel="KL Divergence  ↓")
    ax.set_xticks(K_VALS)
    ax.yaxis.set_major_locator(MaxNLocator(6))
    ax.set_ylim(bottom=0)
    ax.legend(loc="center right")
    fig.tight_layout()
    savefig(fig, "fig1_kl_transfer.png")
    plt.show()


def fig_tv_transfer():
    """Fig 2 — TV distance vs K (with honest annotation of the slight uptick)."""
    fig, ax = plt.subplots(figsize=(6.5, 4.6))
    k_x = np.array(K_VALS)

    ax.fill_between(k_x,
        np.array(TV_MEANS) - np.array(TV_STDS),
        np.array(TV_MEANS) + np.array(TV_STDS),
        color="#FEE4C8", alpha=0.55)
    ax.axhline(ZERO_TV, color=C_RED, linestyle="--", linewidth=1.8,
               label=f"Zero-shot  {ZERO_TV:.4f}", zorder=2)
    ax.errorbar(k_x, TV_MEANS, yerr=TV_STDS,
                fmt="s-", color=C_ORANGE, linewidth=2.2, markersize=7, capsize=4,
                markerfacecolor="white", markeredgewidth=2,
                label="Few-shot adaptation", zorder=3)

    # Annotate the K=20 slight uptick honestly
    ax.annotate("TV slightly\nincreases at K=20\n(KL still improves)",
                xy=(K_VALS[-1], TV_MEANS[-1]),
                xytext=(-60, -30), textcoords="offset points",
                fontsize=7.5, color=C_SUB,
                arrowprops=dict(arrowstyle="->", color=C_SUB, lw=0.8))

    style_ax(ax,
             title="Cross-Device Transfer — Total Variation Distance vs K",
             xlabel="Number of target-device samples  (K)",
             ylabel="Total Variation Distance  ↓")
    ax.set_xticks(K_VALS)
    ax.legend(loc="upper right")
    fig.tight_layout()
    savefig(fig, "fig2_tv_transfer.png")
    plt.show()


def fig_ablation():
    """Fig 3 — Feature ablation. All non-baseline deltas are negative (mismatch features)."""
    fig, ax = plt.subplots(figsize=(7.4, 4.8))

    # Color scheme: baseline = blue; delta < 0 (removes mismatch) = green; delta > 0 (removes useful) = red
    EPS = 1e-3
    colors = []
    for i, (name, delta) in enumerate(zip(ABL_ORDER, ABL_DELTAS)):
        if i == 0:
            colors.append(C_BLUE)
        elif delta < -EPS:
            colors.append(C_GREEN)   # removing it helps → was a mismatch feature
        elif delta > EPS:
            colors.append(C_RED)     # removing it hurts → was a useful feature
        else:
            colors.append(C_AMBER)   # negligible

    bars = ax.bar(range(len(ABL_ORDER)), ABL_VALS,
                  color=colors, edgecolor="white", linewidth=0.8,
                  width=0.55, zorder=3)
    ax.axhline(baseline_kl, color=C_BLUE, linestyle="--", linewidth=1.3,
               alpha=0.6, label=f"Baseline (all features) = {baseline_kl:.4f}")

    ax.set_xticks(range(len(ABL_ORDER)))
    ax.set_xticklabels(["All\nfeatures\n(baseline)", "Remove\nT1", "Remove\nT2",
                         "Remove\nReadout\nError", "Remove\nCX Gate\nError"], fontsize=9)
    add_bar_values(ax, fmt="{:.4f}", fontsize=8)

    from matplotlib.patches import Patch
    legend_patches = [
        Patch(color=C_BLUE,  label="Full model baseline"),
        Patch(color=C_RED,   label="Removal hurts  (useful feature)"),
        Patch(color=C_GREEN, label="Removal helps  (cross-device mismatch)"),
        Patch(color=C_AMBER, label="Negligible effect"),
    ]
    ax.legend(handles=legend_patches, fontsize=8, loc="center right")
    style_ax(ax, title="Feature Ablation Study — Impact on Zero-Shot KL Divergence\n(negative Δ indicates removal improves transfer)",
             ylabel="KL Divergence  ↓")
    ax.set_ylim(bottom=0)
    fig.tight_layout()
    savefig(fig, "fig3_ablation.png")
    plt.show()


def fig_training():
    """Fig 4 — Training curve (real data from training_history.json)."""
    fig, ax = plt.subplots(figsize=(6.5, 4.6))
    ep = np.arange(len(train_loss))

    best_epoch_display = best_epoch + 1
    stopped_display = stopped + 1

    ax.plot(ep, train_loss, color=C_PURPLE, linewidth=2.0, label="Train KL")
    ax.plot(ep, val_loss,   color=C_TEAL, linewidth=1.8,
            linestyle="--", label="Validation KL")

    ax.axvline(best_epoch, color=C_GREEN, linestyle=":", linewidth=1.4,
               alpha=0.8, label=f"Best epoch {best_epoch_display}")
    ax.axvline(stopped, color=C_RED, linestyle=":", linewidth=1.2,
               alpha=0.6, label=f"Early stop ep. {stopped_display}")

    ax.text(0.97, 0.97,
            f"Best val KL = {best_val:.4f}\nep. {best_epoch_display}  |  stop ep. {stopped_display}",
            transform=ax.transAxes, fontsize=8.5, color=C_SUB,
            ha="right", va="top",
            bbox=dict(boxstyle="round,pad=0.35", facecolor=C_PANEL,
                      edgecolor=C_BORDER, linewidth=0.8))

    ax.set_yscale("log")
    style_ax(ax, title=f"Training Convergence  —  {BACKEND_A}",
             xlabel="Epoch", ylabel="KL Divergence Loss  (log scale)  ↓")
    ax.legend(fontsize=8, loc="lower left")
    fig.tight_layout()
    savefig(fig, "fig4_training.png")
    plt.show()


def fig_example_prediction():
    """Fig 5 — Distribution comparison + per-sample metrics panel."""
    fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.8),
                             gridspec_kw={"width_ratios": [2.2, 1]})

    # ── Left: grouped bars ──
    ax = axes[0]
    idx = np.arange(N_STATES)
    w   = 0.26
    ax.bar(idx - w, EX_NOISY, width=w, color="#FC8181", alpha=0.9,
           label="Noisy (input)", zorder=3)
    ax.bar(idx,     EX_IDEAL, width=w, color=C_BLUE, alpha=0.85,
           label="Ideal (target)", zorder=3)
    ax.bar(idx + w, EX_PRED,  width=w, color=C_GREEN, alpha=0.85,
           label="Model prediction", zorder=3)

    style_ax(ax,
             title=f"Probability Distribution Comparison\n"
                   f"{EX_TYPE.title()} circuit  ·  {EX_N} qubits  ·  Backend B",
             xlabel="Bitstring state",
             ylabel="Probability")
    ax.set_xticks(idx)
    ax.set_xticklabels(STATE_LABELS, fontsize=8.5)
    ax.legend(loc="upper right", fontsize=9)

    # ── Right: metric summary ──
    ax2 = axes[1]
    ax2.set_axis_off()
    ax2.set_title("Per-Sample Metrics", fontsize=10, fontweight="bold", pad=8)

    kl_reduction = 100.0*(EX_KL_NOISY - EX_KL_PRED)/EX_KL_NOISY if EX_KL_NOISY > 0 else 0
    tv_reduction = 100.0*(EX_TV_NOISY - EX_TV_PRED)/EX_TV_NOISY if EX_TV_NOISY > 0 else 0

    rows = [
        ("KL:  noisy → ideal",    f"{EX_KL_NOISY:.4f}", C_RED),
        ("KL:  pred  → ideal",    f"{EX_KL_PRED:.4f}",  C_GREEN),
        ("KL reduction",          f"{kl_reduction:.1f}%", C_BLUE),
        ("TV:  noisy → ideal",    f"{EX_TV_NOISY:.4f}", C_RED),
        ("TV:  pred  → ideal",    f"{EX_TV_PRED:.4f}",  C_GREEN),
        ("TV reduction",          f"{tv_reduction:.1f}%", C_BLUE),
    ]
    y = 0.90
    for label, val, color in rows:
        ax2.text(0.05, y, label, transform=ax2.transAxes,
                 fontsize=8.5, color=C_SUB, va="top")
        ax2.text(0.95, y, val, transform=ax2.transAxes,
                 fontsize=9, color=color, va="top", ha="right", fontweight="bold")
        y -= 0.04
        ax2.plot([0.02, 0.98], [y + 0.005, y + 0.005],
                 color=C_GRID, linewidth=0.6, transform=ax2.transAxes, clip_on=False)
        y -= 0.10

    fig.tight_layout()
    savefig(fig, "fig5_example_prediction.png")
    plt.show()


def fig_results_table():
    """Fig 6 — Clean results table with both improvement metrics."""
    fig, ax = plt.subplots(figsize=(8.2, 4.2))
    ax.set_axis_off()

    rows = [
        ["In-domain (A→A)", f"{IN_KL:.4f}", "—", "—", "—"],
        ["Zero-shot (A→B)", f"{ZERO_KL:.4f}", f"{ZERO_TV:.4f}", "baseline", "—"],
    ]
    for k, km, ks, tm, ri, gi in zip(
            K_VALS, KL_MEANS, KL_STDS, TV_MEANS, IMPROV_REL, IMPROV_GAP):
        rows.append([
            f"Few-shot  K={k}",
            f"{km:.4f} ± {ks:.4f}",
            f"{tm:.4f}",
            f"−{ri:.1f}%",
            f"−{gi:.1f}%",
        ])

    col_labels = ["Condition", "KL Divergence ↓",
                  "TV Distance ↓", "Δ KL (vs zero-shot)", "Gap recovery (%)"]

    tbl = ax.table(
        cellText=rows, colLabels=col_labels,
        cellLoc="center", loc="center",
        bbox=[0, 0, 1, 1])
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(9)
    tbl.scale(1, 2.1)

    for (r, c), cell in tbl.get_celld().items():
        cell.set_linewidth(0.6)
        cell.set_edgecolor(C_BORDER)
        if r == 0:
            cell.set_facecolor(C_SLATE)
            cell.get_text().set_color("white")
            cell.get_text().set_weight("bold")
        elif r == 1:           # in-domain
            cell.set_facecolor("#EBF8FF")
        elif r == 2:           # zero-shot
            cell.set_facecolor("#FFF5F5")
        else:
            cell.set_facecolor(C_PANEL if r % 2 == 0 else C_LIGHT)
        if c in (3, 4) and r > 2:
            cell.get_text().set_color(C_GREEN)
            cell.get_text().set_weight("bold")

    ax.set_title("Results Summary", fontsize=11, fontweight="bold", pad=14)
    fig.tight_layout()
    savefig(fig, "fig6_results_table.png")
    plt.show()


def fig_dataset_overview():
    """Fig 7 — Dataset composition (requires dataset.json)."""
    if not has_dataset:
        print("  ⚠  dataset.json not found — skipping Fig 7")
        return

    from collections import Counter
    backend_counts = Counter(d["backend"]       for d in dataset)
    type_counts    = Counter(d["circuit_type"]  for d in dataset)
    qubit_counts   = Counter(d["n_qubits"]      for d in dataset)

    fig, axes = plt.subplots(1, 3, figsize=(11.5, 4.4))

    ax = axes[0]
    ax.bar(list(backend_counts.keys()), list(backend_counts.values()),
           color=PALETTE[:2], edgecolor="white", linewidth=0.8,
           width=0.5, zorder=3)
    style_ax(ax, title="By Backend", ylabel="Circuit count")
    add_bar_values(ax, fmt="{:.0f}", fontsize=9)

    ax = axes[1]
    t_names = sorted(type_counts.keys())
    ax.bar(t_names, [type_counts[k] for k in t_names],
           color=PALETTE[:len(t_names)], edgecolor="white", linewidth=0.8,
           width=0.55, zorder=3)
    style_ax(ax, title="By Circuit Type", ylabel="Circuit count")
    add_bar_values(ax, fmt="{:.0f}", fontsize=9)
    ax.tick_params(axis="x", labelsize=8.5)

    ax = axes[2]
    q_names = sorted(qubit_counts.keys())
    ax.bar(q_names, [qubit_counts[k] for k in q_names],
           color=C_TEAL, edgecolor="white", linewidth=0.8,
           width=0.55, zorder=3)
    style_ax(ax, title="By Qubit Count", xlabel="Qubits", ylabel="Circuit count")
    add_bar_values(ax, fmt="{:.0f}", fontsize=9)

    fig.suptitle("Dataset Composition  (170 total samples)",
                 fontsize=12, fontweight="bold", y=1.01)
    fig.tight_layout()
    savefig(fig, "fig7_dataset_overview.png")
    plt.show()


def fig_calibration_drift():
    """Fig 8 — Device calibration comparison: coherence + errors."""
    fig, axes = plt.subplots(1, 2, figsize=(9.8, 4.6))
    w = 0.32

    ax = axes[0]
    x  = np.arange(2)
    ax.bar(x - w/2, [A_T1, A_T2], width=w, color=C_BLUE,
           label=BACKEND_A, edgecolor="white", linewidth=0.7, zorder=3)
    ax.bar(x + w/2, [B_T1, B_T2], width=w, color=C_ORANGE,
           label=BACKEND_B, edgecolor="white", linewidth=0.7, zorder=3)
    ax.set_xticks(x)
    ax.set_xticklabels(["T₁ relaxation (µs)", "T₂ coherence (µs)"])
    for i, (a, b) in enumerate([(A_T1, B_T1), (A_T2, B_T2)]):
        delta_pct = 100*(b - a)/a
        ax.text(i, max(a, b) + 3, f"+{delta_pct:.0f}%",
                ha="center", fontsize=8.5, color=C_ORANGE, fontweight="bold")
    style_ax(ax, title="Coherence Times", ylabel="Microseconds (µs)")
    ax.legend(loc="upper left", fontsize=9)

    ax = axes[1]
    x  = np.arange(2)
    ax.bar(x - w/2, [A_RO, A_CX], width=w, color=C_BLUE,
           label=BACKEND_A, edgecolor="white", linewidth=0.7, zorder=3)
    ax.bar(x + w/2, [B_RO, B_CX], width=w, color=C_ORANGE,
           label=BACKEND_B, edgecolor="white", linewidth=0.7, zorder=3)
    ax.set_xticks(x)
    ax.set_xticklabels(["Readout error", "CX gate error"])
    for i, (a, b) in enumerate([(A_RO, B_RO), (A_CX, B_CX)]):
        delta_pct = 100*(b - a)/a
        ax.text(i, max(a, b)*1.04, f"+{delta_pct:.0f}%",
                ha="center", fontsize=8.5, color=C_ORANGE, fontweight="bold")
    style_ax(ax, title="Error Rates", ylabel="Error probability")
    ax.legend(loc="upper left", fontsize=9)

    fig.suptitle(f"Device Calibration Drift  —  {BACKEND_A}  vs  {BACKEND_B}",
                 fontsize=12, fontweight="bold", y=1.01)
    fig.tight_layout()
    savefig(fig, "fig8_calibration_drift.png")
    plt.show()


# ── MAIN DASHBOARD ────────────────────────────────────────────
def fig_dashboard():
    """
    3-row publication dashboard.
    Row 0 : 4 stat cards
    Row 1 : KL curve | TV curve | Ablation
    Row 2 : Calibration drift | Results table | Training curve
    """
    fig = plt.figure(figsize=(19, 12.5), facecolor=C_BG)
    outer = gridspec.GridSpec(3, 1, figure=fig,
                              height_ratios=[0.68, 4.0, 4.0],
                              hspace=0.52)
    row0 = gridspec.GridSpecFromSubplotSpec(1, 4, subplot_spec=outer[0], wspace=0.28)
    row1 = gridspec.GridSpecFromSubplotSpec(1, 3, subplot_spec=outer[1], wspace=0.32)
    row2 = gridspec.GridSpecFromSubplotSpec(1, 3, subplot_spec=outer[2], wspace=0.32)

    # ── Row 0: stat cards ──
    draw_stat_card(fig.add_subplot(row0[0,0]),
                   "In-domain KL  (A→A)", f"{IN_KL:.4f}",
                   sub="Upper-bound reference", accent=C_GREEN)
    draw_stat_card(fig.add_subplot(row0[0,1]),
                   "Zero-shot KL  (A→B)", f"{ZERO_KL:.4f}",
                   sub=f"{ZERO_KL/IN_KL:.1f}× in-domain degradation", accent=C_RED)
    draw_stat_card(fig.add_subplot(row0[0,2]),
                   f"Best few-shot KL  (K={BEST_K})", f"{BEST_KL:.4f}",
                   sub=f"−{IMPROV_REL[BEST_K_IDX]:.1f}% vs zero-shot", accent=C_BLUE)
    draw_stat_card(fig.add_subplot(row0[0,3]),
                   f"Gap recovery  (K={K_VALS[-1]})",
                   f"{IMPROV_GAP[-1]:.1f}%",
                   sub=f"({IMPROV_REL[-1]:.1f}% vs zero-shot baseline)", accent=C_AMBER)

    # ── Row 1, col 0: KL curve ──
    ax = fig.add_subplot(row1[0,0])
    k_x = np.array(K_VALS)
    ax.fill_between(k_x,
        np.array(KL_MEANS)-np.array(KL_STDS),
        np.array(KL_MEANS)+np.array(KL_STDS),
        color=C_BLUE_L, alpha=0.45)
    ax.axhline(ZERO_KL, color=C_RED, linestyle="--", linewidth=1.6,
               label=f"Zero-shot  {ZERO_KL:.4f}")
    ax.axhline(IN_KL, color=C_GREEN, linestyle=":", linewidth=1.4,
               label=f"In-domain  {IN_KL:.4f}")
    ax.errorbar(k_x, KL_MEANS, yerr=KL_STDS,
                fmt="o-", color=C_BLUE, linewidth=2.0, markersize=6,
                capsize=3.5, markerfacecolor="white", markeredgewidth=1.8)
    for k, km, ri in zip(K_VALS, KL_MEANS, IMPROV_REL):
        ax.annotate(f"−{ri:.1f}%", xy=(k, km), xytext=(0, 9),
                    textcoords="offset points",
                    ha="center", fontsize=7.5, color=C_BLUE, fontweight="bold")
    style_ax(ax, title="KL Divergence vs Few-Shot K",
             xlabel="K (target-device samples)", ylabel="KL Divergence  ↓")
    ax.set_xticks(K_VALS); ax.legend(fontsize=8)

    # ── Row 1, col 1: TV curve ──
    ax = fig.add_subplot(row1[0,1])
    ax.fill_between(k_x,
        np.array(TV_MEANS)-np.array(TV_STDS),
        np.array(TV_MEANS)+np.array(TV_STDS),
        color="#FEE4C8", alpha=0.55)
    ax.axhline(ZERO_TV, color=C_RED, linestyle="--", linewidth=1.6,
               label=f"Zero-shot  {ZERO_TV:.4f}")
    ax.errorbar(k_x, TV_MEANS, yerr=TV_STDS,
                fmt="s-", color=C_ORANGE, linewidth=2.0, markersize=6,
                capsize=3.5, markerfacecolor="white", markeredgewidth=1.8,
                label="Few-shot")
    ax.annotate("slight TV\nuptick at K=20\n(see §5.1)",
                xy=(K_VALS[-1], TV_MEANS[-1]),
                xytext=(-50, -28), textcoords="offset points",
                fontsize=7, color=C_SUB,
                arrowprops=dict(arrowstyle="->", color=C_SUB, lw=0.7))
    style_ax(ax, title="TV Distance vs Few-Shot K",
             xlabel="K", ylabel="Total Variation Distance  ↓")
    ax.set_xticks(K_VALS); ax.legend(fontsize=8)

    # ── Row 1, col 2: Ablation ──
    ax = fig.add_subplot(row1[0,2])
    EPS = 1e-3
    colors = [C_BLUE if i==0 else
              (C_GREEN if d < -EPS else C_RED if d > EPS else C_AMBER)
              for i, d in enumerate(ABL_DELTAS)]
    ax.bar(range(len(ABL_ORDER)), ABL_VALS,
           color=colors, edgecolor="white", linewidth=0.7, width=0.55, zorder=3)
    ax.axhline(baseline_kl, color=C_BLUE, linestyle="--", linewidth=1.2, alpha=0.6)
    ax.set_xticks(range(len(ABL_ORDER)))
    ax.set_xticklabels(["All\nfeatures", "−T1", "−T2", "−Readout\nError",
                         "−CX\nGate Err"], fontsize=8.5)
    add_bar_values(ax, fmt="{:.3f}", fontsize=7.5)
    style_ax(ax, title="Feature Ablation Study", ylabel="KL Divergence  ↓")

    # ── Row 2, col 0: Calibration drift ──
    ax = fig.add_subplot(row2[0,0])
    cat = ["T₁ (µs)", "T₂ (µs)", "Readout\nerr ×100", "CX gate\nerr ×100"]
    va = [A_T1, A_T2, A_RO*100, A_CX*100]
    vb = [B_T1, B_T2, B_RO*100, B_CX*100]
    xp = np.arange(4); wp = 0.30
    ax.bar(xp - wp/2, va, width=wp, color=C_BLUE, label=BACKEND_A,
           edgecolor="white", linewidth=0.7, zorder=3)
    ax.bar(xp + wp/2, vb, width=wp, color=C_ORANGE, label=BACKEND_B,
           edgecolor="white", linewidth=0.7, zorder=3)
    ax.set_xticks(xp); ax.set_xticklabels(cat, fontsize=8.5)
    style_ax(ax, title="Calibration Drift (A vs B)",
             ylabel="Value (µs  or  ×100 for errors)")
    ax.legend(fontsize=8)

    # ── Row 2, col 1: Results table ──
    ax = fig.add_subplot(row2[0,1])
    ax.set_axis_off()
    tbl_rows = [
        ["In-domain", f"{IN_KL:.4f}", "—", "—"],
        ["Zero-shot", f"{ZERO_KL:.4f}", f"{ZERO_TV:.4f}", "baseline"],
    ]
    for k, km, ks, tm, ri in zip(K_VALS, KL_MEANS, KL_STDS, TV_MEANS, IMPROV_REL):
        tbl_rows.append([f"K={k}", f"{km:.4f}±{ks:.3f}", f"{tm:.4f}", f"−{ri:.1f}%"])
    tbl = ax.table(cellText=tbl_rows,
                   colLabels=["Condition", "KL ↓", "TV ↓", "Δ KL"],
                   cellLoc="center", loc="center", bbox=[0,0,1,1])
    tbl.auto_set_font_size(False); tbl.set_fontsize(8.5); tbl.scale(1, 2.05)
    for (r, c), cell in tbl.get_celld().items():
        cell.set_linewidth(0.5); cell.set_edgecolor(C_BORDER)
        if r == 0:
            cell.set_facecolor(C_SLATE)
            cell.get_text().set_color("white")
            cell.get_text().set_weight("bold")
        elif r == 1:  cell.set_facecolor("#EBF8FF")
        elif r == 2:  cell.set_facecolor("#FFF5F5")
        else:  cell.set_facecolor(C_PANEL if r%2==0 else C_LIGHT)
        if c == 3 and r > 2:
            cell.get_text().set_color(C_GREEN)
            cell.get_text().set_weight("bold")
    ax.set_title("Results Summary", fontsize=10, fontweight="bold", pad=10)

    # ── Row 2, col 2: Training curve ──
    ax = fig.add_subplot(row2[0,2])
    ep = np.arange(len(train_loss))
    ax.plot(ep, train_loss, color=C_PURPLE, linewidth=1.8, label="Train KL")
    ax.plot(ep, val_loss,   color=C_TEAL,   linewidth=1.6,
            linestyle="--", label="Val KL")
    ax.axvline(best_epoch, color=C_GREEN, linestyle=":", linewidth=1.2, alpha=0.8)
    ax.set_yscale("log")
    ax.text(0.97, 0.95,
            f"Best val = {best_val:.4f}\nep. {best_epoch}  |  stop ep. {stopped}",
            transform=ax.transAxes, fontsize=8, color=C_SUB,
            ha="right", va="top",
            bbox=dict(boxstyle="round,pad=0.25", facecolor=C_PANEL,
                      edgecolor=C_BORDER, linewidth=0.6))
    style_ax(ax, title=f"Training Convergence ({BACKEND_A})",
             xlabel="Epoch", ylabel="KL Loss  (log scale)")
    ax.legend(fontsize=8)

    fig.suptitle("Quantum Noise Few-Shot Transfer Learning — Experimental Results",
                 fontsize=16, fontweight="bold", y=0.998)
    fig.tight_layout(rect=[0, 0, 1, 0.997])
    savefig(fig, "dashboard.png", dpi=300)

    pdf_path = os.path.join(OUTDIR, "dashboard.pdf")
    fig.savefig(pdf_path, dpi=300)
    print(f"  ✓  {pdf_path}")
    plt.show()


def fig_supplementary():
    """Supplementary overview page."""
    fig, axes = plt.subplots(2, 3, figsize=(16, 9.5))

    # 1. KL all conditions
    ax = axes[0, 0]
    cond_names = ["In-domain", "Zero-shot"] + [f"K={k}" for k in K_VALS]
    cond_vals  = [IN_KL, ZERO_KL] + KL_MEANS
    cond_cols  = [C_GREEN, C_RED] + [C_BLUE]*len(K_VALS)
    ax.bar(cond_names, cond_vals, color=cond_cols,
           edgecolor="white", linewidth=0.8, width=0.55, zorder=3)
    style_ax(ax, title="KL Performance — All Conditions", ylabel="KL Divergence  ↓")
    add_bar_values(ax, fmt="{:.4f}", fontsize=8.5)

    # 2. Dual improvement metrics
    ax = axes[0, 1]
    x = np.arange(len(K_VALS)); w = 0.35
    ax.bar(x - w/2, IMPROV_REL, width=w, color=C_BLUE,   label="vs zero-shot (%)", zorder=3)
    ax.bar(x + w/2, IMPROV_GAP, width=w, color=C_AMBER, label="gap recovery (%)", zorder=3)
    ax.set_xticks(x); ax.set_xticklabels([f"K={k}" for k in K_VALS])
    style_ax(ax, title="Improvement Metrics at Each K", ylabel="Improvement (%)")
    ax.legend(fontsize=8.5)
    add_bar_values(ax, fmt="{:.1f}", fontsize=8)

    # 3. Calibration delta (B − A)
    ax = axes[0, 2]
    delta_names = ["T₁", "T₂", "Readout\nerror", "CX gate\nerror"]
    delta_vals  = [B_T1-A_T1, B_T2-A_T2, B_RO-A_RO, B_CX-A_CX]
    colors_d    = [C_ORANGE if d > 0 else C_BLUE for d in delta_vals]
    ax.bar(delta_names, delta_vals, color=colors_d,
           edgecolor="white", linewidth=0.8, width=0.5, zorder=3)
    ax.axhline(0, color=C_SUB, linewidth=0.8)
    style_ax(ax, title=f"Calibration Delta  (B − A)", ylabel="Difference")
    add_bar_values(ax, fmt="{:.4f}", fontsize=8.5)

    # 4. Ablation delta view
    ax = axes[1, 0]
    delta_only = ABL_DELTAS[1:]   # exclude baseline
    names_only = [k.replace("Remove ", "−") for k in ABL_ORDER[1:]]
    d_colors = [C_RED if d > 1e-3 else C_GREEN for d in delta_only]
    ax.bar(names_only, delta_only, color=d_colors,
           edgecolor="white", linewidth=0.8, width=0.5, zorder=3)
    ax.axhline(0, color=C_SUB, linewidth=0.8)
    style_ax(ax, title="Ablation Δ KL (vs all-features baseline)",
             ylabel="Δ KL  (negative = mismatch feature)")
    add_bar_values(ax, fmt="{:+.4f}", fontsize=8.5)

    # 5. TV all conditions
    ax = axes[1, 1]
    tv_names = ["Zero-shot"] + [f"K={k}" for k in K_VALS]
    tv_vals  = [ZERO_TV] + TV_MEANS
    tv_cols  = [C_RED] + [C_ORANGE]*len(K_VALS)
    ax.bar(tv_names, tv_vals, color=tv_cols,
           edgecolor="white", linewidth=0.8, width=0.5, zorder=3)
    style_ax(ax, title="TV Distance — All Conditions", ylabel="Total Variation  ↓")
    add_bar_values(ax, fmt="{:.4f}", fontsize=8.5)
    ax.text(0.98, 0.98, "Note: TV slightly\nincreases at K=20\n(see §5.1)",
            transform=ax.transAxes, fontsize=8, color=C_SUB,
            ha="right", va="top",
            bbox=dict(boxstyle="round,pad=0.3", facecolor=C_PANEL,
                      edgecolor=C_BORDER, linewidth=0.6))

    # 6. Training detail
    ax = axes[1, 2]
    ep = np.arange(len(train_loss))
    ax.plot(ep, train_loss, color=C_PURPLE, linewidth=1.8, label="Train KL")
    ax.plot(ep, val_loss,   color=C_TEAL, linewidth=1.6, linestyle="--", label="Val KL")
    ax.axvline(best_epoch, color=C_GREEN, linestyle=":", linewidth=1.4,
               label=f"Best ep. {best_epoch}")
    ax.set_yscale("log")
    style_ax(ax, title="Training Detail  (log scale)",
             xlabel="Epoch", ylabel="KL Loss")
    ax.legend(fontsize=8.5)

    fig.suptitle("Supplementary Experimental Overview",
                 fontsize=14, fontweight="bold", y=1.01)
    fig.tight_layout()
    savefig(fig, "supplementary_summary.png", dpi=300)
    plt.show()


# ============================================================
# ── RUN ALL ──────────────────────────────────────────────────
# ============================================================
print("\n" + "="*55)
print("  Generating paper-ready figures...")
print("="*55 + "\n")

fig_kl_transfer()
fig_tv_transfer()
fig_ablation()
fig_training()
fig_example_prediction()
fig_results_table()
fig_dataset_overview()
fig_calibration_drift()
fig_dashboard()
fig_supplementary()

# ── Bundle PDF ───────────────────────────────────────────────
pdf_path = os.path.join(OUTDIR, "all_figures.pdf")
figure_files = [
    "dashboard.png",
    "supplementary_summary.png",
    "fig1_kl_transfer.png",
    "fig2_tv_transfer.png",
    "fig3_ablation.png",
    "fig4_training.png",
    "fig5_example_prediction.png",
    "fig6_results_table.png",
    "fig8_calibration_drift.png",
]
if has_dataset:
    figure_files.insert(-1, "fig7_dataset_overview.png")

with PdfPages(pdf_path) as pdf:
    for fname in figure_files:
        fpath = os.path.join(OUTDIR, fname)
        if not os.path.exists(fpath): continue
        img = plt.imread(fpath)
        ftmp = plt.figure(figsize=(14, 9), facecolor=C_BG)
        atmp = ftmp.add_subplot(1, 1, 1)
        atmp.imshow(img, aspect="auto"); atmp.axis("off")
        pdf.savefig(ftmp, bbox_inches="tight", facecolor=C_BG)
        plt.close(ftmp)

print(f"\n  ✓  PDF bundle → {pdf_path}")

# ── Auto-download in Colab ────────────────────────────────────
try:
    import zipfile
    zip_path = "paper_viz_figures.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for root, dirs, files in os.walk(OUTDIR):
            for file in files:
                zf.write(os.path.join(root, file), file)
    from google.colab import files
    files.download(zip_path)
    print(f"  ✓  Download triggered → {zip_path}")
except Exception:
    print(f"  ℹ  Figures saved to ./{OUTDIR}/  (not in Colab)")

print("\n" + "="*55)
print("  ✅  All done!")
print("="*55)

  ✓  Loaded all results
     Backend A : ibm_fez  |  Backend B : ibm_marrakesh
     In-domain KL = 0.3014
     Zero-shot KL = 1.6706  (5.5× degradation)
     K=20 KL      = 1.1924  (28.6% vs zero-shot, 34.9% gap recovery)

  Generating paper-ready figures...

  ✓  paper_viz/fig1_kl_transfer.png
  ✓  paper_viz/fig2_tv_transfer.png
  ✓  paper_viz/fig3_ablation.png
  ✓  paper_viz/fig4_training.png
  ✓  paper_viz/fig5_example_prediction.png
  ✓  paper_viz/fig6_results_table.png
  ✓  paper_viz/fig7_dataset_overview.png
  ✓  paper_viz/fig8_calibration_drift.png
  ✓  paper_viz/dashboard.png
  ✓  paper_viz/dashboard.pdf
  ✓  paper_viz/supplementary_summary.png

  ✓  PDF bundle → paper_viz/all_figures.pdf


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  ✓  Download triggered → paper_viz_figures.zip

  ✅  All done!
